# 00_setup — Catálogo, schemas, volume, máscaras y permisos
Ejecutar **una vez por target** (`electrocasa_dev` y, si corresponde, `electrocasa_prod`) **antes** de desplegar el bundle.

Prerrequisito: crear en **Settings > Identity and access > Groups** los grupos
`electrocasa_ingenieria`, `electrocasa_analistas` y `electrocasa_auditoria`, y agregarte a `electrocasa_ingenieria`
(el dueño del pipeline tiene que estar en ese grupo para que las tablas derivadas de Empleados guarden los valores reales).

In [ ]:
dbutils.widgets.text("catalog", "electrocasa_dev")
dbutils.widgets.text("managed_location", "")   # solo si tu metastore no tiene storage root (Azure). Ej: abfss://...@<cuenta>.dfs.core.windows.net/electrocasa
CATALOG = dbutils.widgets.get("catalog")
LOCATION = dbutils.widgets.get("managed_location").strip()
ING, ANA, AUD = "electrocasa_ingenieria", "electrocasa_analistas", "electrocasa_auditoria"
print(CATALOG, LOCATION or "(storage por defecto del metastore)")

## 1. Catálogo, schemas y volume de landing

In [ ]:
loc = f" MANAGED LOCATION '{LOCATION}'" if LOCATION else ""
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}{loc} COMMENT 'Proyecto integrador ElectroCasa'")
for s in ["bronze", "silver", "gold"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{s} COMMENT 'Capa {s} del modelo Medallion'")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.default.vol_landing COMMENT 'Landing de archivos por fuente'")

for carpeta in ["ventas", "catalogo", "empleados", "resenas", "devoluciones", "sucursales"]:
    dbutils.fs.mkdirs(f"/Volumes/{CATALOG}/default/vol_landing/{carpeta}")
display(dbutils.fs.ls(f"/Volumes/{CATALOG}/default/vol_landing/"))

## 2. Funciones de máscara (dni y salario)
Solo `electrocasa_ingenieria` ve el valor real. Formato elegido:
- **dni:** se muestran solo los 3 últimos dígitos (`*****397`), suficiente para conciliar sin exponer el documento.
- **salario:** `NULL` (un decimal no puede llevar asteriscos y cualquier rango aproximado seguiría filtrando información).

Las máscaras se declaran en el schema de las tablas del pipeline (cláusula `MASK`), no con `ALTER TABLE`.

In [ ]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {CATALOG}.default.mask_dni(dni STRING)
RETURNS STRING
COMMENT 'Máscara de DNI: valor real solo para electrocasa_ingenieria'
RETURN CASE WHEN is_account_group_member('{ING}') THEN dni
            WHEN dni IS NULL THEN NULL
            ELSE concat('*****', right(dni, 3)) END
""")
spark.sql(f"""
CREATE OR REPLACE FUNCTION {CATALOG}.default.mask_salario(salario DECIMAL(10,2))
RETURNS DECIMAL(10,2)
COMMENT 'Máscara de salario: valor real solo para electrocasa_ingenieria'
RETURN CASE WHEN is_account_group_member('{ING}') THEN salario ELSE NULL END
""")
display(spark.sql(f"SELECT is_account_group_member('{ING}') AS soy_ingenieria, current_user() AS usuario"))

## 3. Permisos por grupo

In [ ]:
def run(sql):
    try:
        spark.sql(sql); print("OK   ", sql)
    except Exception as e:
        print("ERROR", sql, "\n     ", str(e).splitlines()[0])

# Ingeniería: lectura y escritura en bronze, silver y gold + volume de landing
run(f"GRANT USE CATALOG ON CATALOG {CATALOG} TO `{ING}`")
for s in ["default", "bronze", "silver", "gold"]:
    run(f"GRANT USE SCHEMA, SELECT, MODIFY, CREATE TABLE, CREATE MATERIALIZED VIEW, CREATE FUNCTION ON SCHEMA {CATALOG}.{s} TO `{ING}`")
run(f"GRANT READ VOLUME, WRITE VOLUME ON VOLUME {CATALOG}.default.vol_landing TO `{ING}`")

# Analistas: solo lectura sobre gold
run(f"GRANT USE CATALOG ON CATALOG {CATALOG} TO `{ANA}`")
run(f"GRANT USE SCHEMA, SELECT ON SCHEMA {CATALOG}.gold TO `{ANA}`")

# Auditoría: solo lectura sobre gold + evidencia de auditoría y lineage
run(f"GRANT USE CATALOG ON CATALOG {CATALOG} TO `{AUD}`")
run(f"GRANT USE SCHEMA, SELECT ON SCHEMA {CATALOG}.gold TO `{AUD}`")

# Funciones de máscara: los tres grupos pueden ejecutarlas (la función decide qué ven)
for g in [ING, ANA, AUD]:
    run(f"GRANT USE SCHEMA ON SCHEMA {CATALOG}.default TO `{g}`")
    for f in ["mask_dni", "mask_salario"]:
        run(f"GRANT EXECUTE ON FUNCTION {CATALOG}.default.{f} TO `{g}`")

### 3.1 Auditoría sobre tablas de sistema
Si el workspace no permite otorgarlos (falta `MANAGE` sobre el catálogo `system`), el error queda impreso y se usa como
evidencia la pestaña **Lineage** de Catalog Explorer (documentado en el README).

In [ ]:
run(f"GRANT USE CATALOG ON CATALOG system TO `{AUD}`")
run(f"GRANT USE SCHEMA ON SCHEMA system.access TO `{AUD}`")
for t in ["audit", "table_lineage", "column_lineage"]:
    run(f"GRANT SELECT ON TABLE system.access.{t} TO `{AUD}`")

## 4. Verificación de permisos

In [ ]:
for g in [ING, ANA, AUD]:
    print(f"==== {g}")
    display(spark.sql(f"SHOW GRANTS `{g}` ON CATALOG {CATALOG}"))
    display(spark.sql(f"SHOW GRANTS `{g}` ON SCHEMA {CATALOG}.gold"))

## 5. Demostración de REVOKE
Se otorga a `electrocasa_analistas` un privilegio que **no** le corresponde (`SELECT` sobre silver), se revoca y se
muestra `SHOW GRANTS` antes y después. Capturar las tres salidas para el README.

In [ ]:
# ANTES: analistas no tiene nada sobre silver
display(spark.sql(f"SHOW GRANTS `{ANA}` ON SCHEMA {CATALOG}.silver"))

In [ ]:
spark.sql(f"GRANT USE SCHEMA, SELECT ON SCHEMA {CATALOG}.silver TO `{ANA}`")   # privilegio indebido
display(spark.sql(f"SHOW GRANTS `{ANA}` ON SCHEMA {CATALOG}.silver"))

In [ ]:
spark.sql(f"REVOKE USE SCHEMA, SELECT ON SCHEMA {CATALOG}.silver FROM `{ANA}`")
# DESPUÉS: vuelve a quedar vacío
display(spark.sql(f"SHOW GRANTS `{ANA}` ON SCHEMA {CATALOG}.silver"))